# Completion Inspector

Side-by-side comparison of model completions before/after an intervention (e.g., base vs SFT vs GRPO) on the same benchmark example. Points at two `eval_all` run roots and discovers benchmarks/stages automatically.

In [37]:
import json
import re
import pandas as pd
import numpy as np
import html as html_lib
import ipywidgets as widgets
from pathlib import Path
from IPython.display import display, HTML, clear_output

pd.set_option("display.max_columns", 20)

In [38]:
# ── Configuration ──────────────────────────────────────────────────────
# Paste paths to eval_all run roots (the timestamp-level directory).
#   e.g. /share/pierson/matt/UAIR/multirun/2026-03-28_eval_all/00-21-18

# QWEN3.5-9B-SFT
# /share/pierson/matt/UAIR/multirun/2026-03-28_eval_all/00-29-29

# QWEN3.5-9B-BASE
# /share/pierson/matt/UAIR/multirun/2026-03-28_eval_all/00-21-18

# QWEN3.5-9B-GRPOv3
# /share/pierson/matt/UAIR/multirun/2026-03-30_eval_all/16-12-04

RUN_A = "/share/pierson/matt/UAIR/multirun/2026-03-28_eval_all/00-21-18"  # BEFORE (e.g. base / zero-shot)
RUN_B = "/share/pierson/matt/UAIR/multirun/2026-03-28_eval_all/00-29-29"  # AFTER  (e.g. SFT or GRPO)

LABEL_A = "Qwen3.5-9B-Base"
LABEL_B = "Qwen3.5-9B-SFT"

In [39]:
# ── Discovery ──────────────────────────────────────────────────────────
# Scan both run roots for benchmark/stage parquets containing completions.

# Map from (benchmark_key, stage_name) -> relative path from run root to parquet
STAGE_CATALOG = {
    ("goldcoin", "llm_inference_applicability"): "goldcoin/goldcoin_hipaa/outputs/llm_inference_applicability/dataset.parquet",
    ("goldcoin", "llm_inference_compliance"): "goldcoin/goldcoin_hipaa/outputs/llm_inference_compliance/dataset.parquet",
    ("goldcoin", "parse_responses_applicability"): "goldcoin/goldcoin_hipaa/outputs/parse_responses_applicability/dataset.parquet",
    ("goldcoin", "parse_responses_compliance"): "goldcoin/goldcoin_hipaa/outputs/parse_responses_compliance/dataset.parquet",
    ("privacylens", "qa_probe_inference"): "privacylens/privacylens_eval/outputs/qa_probe_inference/results.parquet",
    ("privacylens", "agent_action_inference"): "privacylens/privacylens_eval/outputs/agent_action_inference/results.parquet",
    ("vlm_geoprivacy", "vlm_mcq_inference"): "vlm_geoprivacy/vlm_geoprivacy_bench/outputs/vlm_mcq_inference/dataset.parquet",
    ("vlm_geoprivacy", "parse_mcq"): "vlm_geoprivacy/vlm_geoprivacy_bench/outputs/parse_mcq/dataset.parquet",
}


def _resolve_root(run_path: str) -> Path:
    """Resolve the run root, handling the /0/ multirun subdirectory."""
    p = Path(run_path)
    # If the user pointed at the timestamp dir, the actual outputs are under 0/
    if (p / "0").is_dir():
        return p / "0"
    return p


def discover_stages(root: Path) -> dict[tuple[str, str], Path]:
    """Return {(benchmark, stage): parquet_path} for files that exist under root."""
    found = {}
    for key, rel in STAGE_CATALOG.items():
        full = root / rel
        if full.exists():
            found[key] = full
    return found


assert RUN_A and RUN_B, "Set RUN_A and RUN_B in the configuration cell above."

root_a = _resolve_root(RUN_A)
root_b = _resolve_root(RUN_B)
stages_a = discover_stages(root_a)
stages_b = discover_stages(root_b)

# Only keep stages present in both runs
available = sorted(set(stages_a) & set(stages_b))
assert available, (
    f"No matching stages found.\n"
    f"  A has: {sorted(stages_a)}\n"
    f"  B has: {sorted(stages_b)}"
)

print(f"Run A: {root_a}")
print(f"Run B: {root_b}")
print(f"\nAvailable stages ({len(available)}):")
for bench, stage in available:
    print(f"  {bench} / {stage}")

Run A: /share/pierson/matt/UAIR/multirun/2026-03-28_eval_all/00-21-18/0
Run B: /share/pierson/matt/UAIR/multirun/2026-03-28_eval_all/00-29-29/0

Available stages (6):
  goldcoin / llm_inference_applicability
  goldcoin / llm_inference_compliance
  goldcoin / parse_responses_applicability
  goldcoin / parse_responses_compliance
  privacylens / agent_action_inference
  privacylens / qa_probe_inference


In [40]:
# ── Benchmark config registry ─────────────────────────────────────────

BENCH_CONFIG = {
    ("goldcoin", "llm_inference_applicability"): dict(
        id_cols=["case_id"],
        context_cols=["generate_background"],
        ground_truth_col="ground_truth",
        prediction_col=None,
        has_messages=True,
    ),
    ("goldcoin", "llm_inference_compliance"): dict(
        id_cols=["case_id"],
        context_cols=["generate_background"],
        ground_truth_col="ground_truth",
        prediction_col=None,
        has_messages=True,
    ),
    ("goldcoin", "parse_responses_applicability"): dict(
        id_cols=["case_id"],
        context_cols=["generate_background"],
        ground_truth_col="ground_truth",
        prediction_col="prediction",
        has_messages=True,
    ),
    ("goldcoin", "parse_responses_compliance"): dict(
        id_cols=["case_id"],
        context_cols=["generate_background"],
        ground_truth_col="ground_truth",
        prediction_col="prediction",
        has_messages=True,
    ),
    ("privacylens", "qa_probe_inference"): dict(
        id_cols=["name", "seed", "_qa_axis"],
        context_cols=["vignette"],
        ground_truth_col=None,
        prediction_col="predicted_label",
        has_messages=True,
    ),
    ("privacylens", "agent_action_inference"): dict(
        id_cols=["name", "seed"],
        context_cols=["vignette"],
        ground_truth_col=None,
        prediction_col=None,
        has_messages=True,
    ),
    ("vlm_geoprivacy", "vlm_mcq_inference"): dict(
        id_cols=["image_id"],
        context_cols=[],
        ground_truth_col=None,
        prediction_col=None,
        has_messages=False,
    ),
    ("vlm_geoprivacy", "parse_mcq"): dict(
        id_cols=["image_id"],
        context_cols=[],
        ground_truth_col=None,
        prediction_col=None,
        has_messages=False,
    ),
}

In [41]:
# ── HTML rendering helpers ─────────────────────────────────────────────

def _esc(text) -> str:
    """HTML-escape, handling None/NaN."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return "<em>(empty)</em>"
    return html_lib.escape(str(text))


def _extract_user_prompt(messages) -> str:
    """Pull last user-role message content from the messages array."""
    if messages is None:
        return ""
    msgs = list(messages) if isinstance(messages, np.ndarray) else messages
    user_msgs = [m for m in msgs if m.get("role") == "user"]
    return user_msgs[-1]["content"] if user_msgs else ""


def _collapsible(label: str, content: str, max_open: int = 500) -> str:
    """Wrap in <details> if content is long."""
    escaped = _esc(content)
    inner = f'<pre style="white-space:pre-wrap;font-size:13px;margin:4px 0;">{escaped}</pre>'
    if len(content) > max_open:
        return f'<details><summary>{label} ({len(content):,} chars)</summary>{inner}</details>'
    return f'<strong>{label}</strong>{inner}'


def _usage_str(row: pd.Series) -> str:
    if "usage" not in row.index or row["usage"] is None:
        return ""
    u = row["usage"]
    if isinstance(u, dict):
        return f'prompt={u.get("prompt_tokens","?")} compl={u.get("completion_tokens","?")}'
    return ""


def _serialize_val(v):
    """Make a value JSON-serializable."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, np.ndarray):
        return [_serialize_val(x) for x in v]
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    if isinstance(v, dict):
        return {k: _serialize_val(val) for k, val in v.items()}
    return v


def build_record(df_a: pd.DataFrame, df_b: pd.DataFrame, cfg: dict,
                 bench_key: tuple[str, str], idx: int) -> dict:
    """Build a JSON-serializable dict for the comparison at iloc `idx`."""
    benchmark, stage = bench_key
    row_a = df_a.iloc[idx]
    row_b = df_b.iloc[idx]
    id_cols = [c for c in cfg["id_cols"] if c in df_a.columns and c in df_b.columns]

    record = {
        "iloc": idx,
        "benchmark": benchmark,
        "stage": stage,
        "labels": {LABEL_A: str(RUN_A), LABEL_B: str(RUN_B)},
        "ids": {c: _serialize_val(row_a[c]) for c in id_cols},
    }

    # Prompt
    if cfg["has_messages"] and "messages" in row_a.index:
        record["prompt"] = _extract_user_prompt(row_a["messages"])

    # Context
    for col in cfg["context_cols"]:
        if col in row_a.index and row_a[col]:
            record[col] = _serialize_val(row_a[col])

    # Completions
    record["completion_a"] = _serialize_val(row_a.get("generated_text"))
    record["completion_b"] = _serialize_val(row_b.get("generated_text"))

    # Ground truth
    gt_col = cfg["ground_truth_col"]
    if gt_col and gt_col in row_a.index and row_a[gt_col] is not None:
        record["ground_truth"] = _serialize_val(row_a[gt_col])

    # Parsed predictions
    pred_col = cfg["prediction_col"]
    if pred_col:
        if pred_col in row_a.index:
            record["prediction_a"] = _serialize_val(row_a[pred_col])
        if pred_col in row_b.index:
            record["prediction_b"] = _serialize_val(row_b[pred_col])

    # Usage
    if "usage" in row_a.index and row_a["usage"] is not None:
        record["usage_a"] = _serialize_val(row_a["usage"])
    if "usage" in row_b.index and row_b["usage"] is not None:
        record["usage_b"] = _serialize_val(row_b["usage"])

    # VLM MCQ
    if benchmark == "vlm_geoprivacy" and stage in ("vlm_mcq_inference", "parse_mcq"):
        for i in range(1, 8):
            if f"Q{i}_true" in row_a.index:
                record[f"Q{i}_true"] = _serialize_val(row_a[f"Q{i}_true"])
            if f"Q{i}_pred" in df_a.columns:
                record[f"Q{i}_pred_a"] = _serialize_val(row_a[f"Q{i}_pred"])
            if f"Q{i}_pred" in df_b.columns:
                record[f"Q{i}_pred_b"] = _serialize_val(row_b[f"Q{i}_pred"])

    return record


def _searchable_cols(df_a: pd.DataFrame, cfg: dict) -> list[str]:
    """Return column names that make sense to search through."""
    cols = []
    if cfg["has_messages"] and "messages" in df_a.columns:
        cols.append("prompt")  # virtual column — extracted from messages
    cols.append("generated_text")
    for c in cfg["context_cols"]:
        if c in df_a.columns:
            cols.append(c)
    gt = cfg["ground_truth_col"]
    if gt and gt in df_a.columns:
        cols.append(gt)
    pred = cfg["prediction_col"]
    if pred and pred in df_a.columns:
        cols.append(pred)
    for c in cfg["id_cols"]:
        if c in df_a.columns:
            cols.append(c)
    return cols


def search_rows(df_a: pd.DataFrame, df_b: pd.DataFrame, cfg: dict,
                field: str, query: str) -> list[int]:
    """Return iloc indices where `field` matches `query` (case-insensitive regex)
    in either df_a or df_b.  The special field 'prompt' searches extracted user messages."""
    n = min(len(df_a), len(df_b))
    pat = re.compile(query, re.IGNORECASE)
    hits = []
    for i in range(n):
        texts = []
        for df in (df_a, df_b):
            row = df.iloc[i]
            if field == "prompt":
                texts.append(_extract_user_prompt(row.get("messages")))
            elif field in row.index:
                texts.append(str(row[field] or ""))
        if any(pat.search(t) for t in texts):
            hits.append(i)
    return hits


def render_comparison(df_a: pd.DataFrame, df_b: pd.DataFrame, cfg: dict,
                      bench_key: tuple[str, str], idx: int):
    benchmark, stage = bench_key
    row_a = df_a.iloc[idx]
    row_b = df_b.iloc[idx]

    id_cols = [c for c in cfg["id_cols"] if c in df_a.columns and c in df_b.columns]

    # Header
    id_parts = [f"{c}=<b>{_esc(row_a[c])}</b>" for c in id_cols if c in row_a.index]
    id_html = "&ensp;".join(id_parts) if id_parts else ""
    badge = f'<span style="background:#e0e0e0;padding:2px 8px;border-radius:4px;font-size:12px;">{benchmark} / {stage}</span>'

    parts = [f'<div style="margin-bottom:16px;"><h3 style="margin:0;">Row {idx}&ensp;{id_html}&ensp;{badge}</h3></div>']

    # Prompt
    if cfg["has_messages"] and "messages" in row_a.index:
        prompt_text = _extract_user_prompt(row_a["messages"])
        if prompt_text:
            parts.append(
                f'<div style="background:#f5f5f5;padding:8px 12px;border-radius:6px;margin-bottom:12px;">'
                f'{_collapsible("Prompt", prompt_text)}</div>'
            )

    # Context columns
    for col in cfg["context_cols"]:
        if col in row_a.index and row_a[col]:
            parts.append(
                f'<div style="background:#fafafa;padding:6px 12px;border-radius:6px;margin-bottom:8px;">'
                f'{_collapsible(col, str(row_a[col]))}</div>'
            )

    # Side-by-side completions
    text_a = str(row_a.get("generated_text", "") or "")
    text_b = str(row_b.get("generated_text", "") or "")
    usage_a = _usage_str(row_a)
    usage_b = _usage_str(row_b)

    parts.append(f'''
    <table style="width:100%;border-collapse:collapse;margin-bottom:12px;">
    <tr>
      <th style="width:50%;text-align:left;padding:6px 10px;background:#fff3e0;border:1px solid #ffe0b2;">
        {_esc(LABEL_A)} {f'<span style="font-weight:normal;font-size:11px;color:#888;">({usage_a})</span>' if usage_a else ''}
      </th>
      <th style="width:50%;text-align:left;padding:6px 10px;background:#e8f5e9;border:1px solid #c8e6c9;">
        {_esc(LABEL_B)} {f'<span style="font-weight:normal;font-size:11px;color:#888;">({usage_b})</span>' if usage_b else ''}
      </th>
    </tr>
    <tr>
      <td style="vertical-align:top;padding:10px;background:#fffaf0;border:1px solid #ffe0b2;">
        <pre style="white-space:pre-wrap;font-size:13px;margin:0;">{_esc(text_a)}</pre>
      </td>
      <td style="vertical-align:top;padding:10px;background:#f1f8e9;border:1px solid #c8e6c9;">
        <pre style="white-space:pre-wrap;font-size:13px;margin:0;">{_esc(text_b)}</pre>
      </td>
    </tr>
    </table>''')

    # Ground truth
    gt_col = cfg["ground_truth_col"]
    if gt_col and gt_col in row_a.index and row_a[gt_col] is not None:
        parts.append(
            f'<div style="background:#e3f2fd;padding:6px 12px;border-radius:6px;margin-bottom:8px;">'
            f'<strong>Ground truth:</strong> {_esc(row_a[gt_col])}</div>'
        )

    # Parsed predictions (side by side)
    pred_col = cfg["prediction_col"]
    if pred_col:
        pred_a = row_a.get(pred_col) if pred_col in row_a.index else None
        pred_b = row_b.get(pred_col) if pred_col in row_b.index else None
        if pred_a is not None or pred_b is not None:
            parts.append(
                f'<div style="margin-bottom:8px;">'
                f'<strong>Parsed prediction</strong> ({pred_col}): '
                f'<span style="background:#fff3e0;padding:2px 6px;border-radius:3px;">{_esc(pred_a)}</span> vs '
                f'<span style="background:#e8f5e9;padding:2px 6px;border-radius:3px;">{_esc(pred_b)}</span></div>'
            )

    # VLM MCQ Q1-Q7 grid
    if benchmark == "vlm_geoprivacy" and stage in ("vlm_mcq_inference", "parse_mcq"):
        q_true = [f"Q{i}_true" for i in range(1, 8) if f"Q{i}_true" in row_a.index]
        q_pred_a = [f"Q{i}_pred" for i in range(1, 8) if f"Q{i}_pred" in df_a.columns]
        q_pred_b = [f"Q{i}_pred" for i in range(1, 8) if f"Q{i}_pred" in df_b.columns]
        if q_true or q_pred_a or q_pred_b:
            rows_html = ""
            for i in range(1, 8):
                tv = _esc(row_a.get(f"Q{i}_true", "")) if f"Q{i}_true" in row_a.index else "-"
                pa = _esc(row_a.get(f"Q{i}_pred", "")) if f"Q{i}_pred" in df_a.columns else "-"
                pb = _esc(row_b.get(f"Q{i}_pred", "")) if f"Q{i}_pred" in df_b.columns else "-"
                rows_html += f"<tr><td style='padding:2px 8px;'>Q{i}</td><td style='padding:2px 8px;'>{tv}</td><td style='padding:2px 8px;background:#fffaf0;'>{pa}</td><td style='padding:2px 8px;background:#f1f8e9;'>{pb}</td></tr>"
            parts.append(
                f'<table style="border-collapse:collapse;margin-bottom:8px;">'
                f'<tr><th style="padding:2px 8px;">Q</th><th style="padding:2px 8px;">True</th>'
                f'<th style="padding:2px 8px;background:#fff3e0;">{_esc(LABEL_A)}</th>'
                f'<th style="padding:2px 8px;background:#e8f5e9;">{_esc(LABEL_B)}</th></tr>'
                f'{rows_html}</table>'
            )

    display(HTML("\n".join(parts)))

In [42]:
# ── Interactive browser ────────────────────────────────────────────────

# State: currently loaded dataframes and search results
_state = {"df_a": None, "df_b": None, "key": None, "cfg": None, "filtered_idxs": None}

# ── Widgets ───────────────────────────────────────────────────────────
stage_dropdown = widgets.Dropdown(
    options=[(f"{b} / {s}", (b, s)) for b, s in available],
    description="Stage:",
    layout=widgets.Layout(width="400px"),
)

idx_slider = widgets.IntSlider(
    value=0, min=0, max=0, step=1,
    description="Row (iloc):",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)
idx_input = widgets.BoundedIntText(
    value=0, min=0, max=0, step=1,
    description="Jump to:",
    layout=widgets.Layout(width="200px"),
)
widgets.jslink((idx_slider, "value"), (idx_input, "value"))

# Search
search_field = widgets.Dropdown(
    options=["generated_text"],
    description="Field:",
    layout=widgets.Layout(width="250px"),
)
search_input = widgets.Text(
    placeholder="regex pattern (case-insensitive)",
    description="Search:",
    layout=widgets.Layout(width="350px"),
)
search_btn = widgets.Button(description="Search", button_style="primary",
                            layout=widgets.Layout(width="80px"))
clear_btn = widgets.Button(description="Clear", layout=widgets.Layout(width="70px"))

# Search result navigation
search_pos_label = widgets.HTML(value="")
search_prev_btn = widgets.Button(description="Prev", layout=widgets.Layout(width="60px"))
search_next_btn = widgets.Button(description="Next", layout=widgets.Layout(width="60px"))
search_nav = widgets.HBox([search_prev_btn, search_next_btn, search_pos_label])
search_nav.layout.display = "none"  # hidden until search is active

# Save
save_btn = widgets.Button(description="Save to JSON", button_style="info",
                           layout=widgets.Layout(width="120px"))
save_status = widgets.HTML(value="")

# Outputs
info_output = widgets.Output()
main_output = widgets.Output()

# ── Helpers ───────────────────────────────────────────────────────────

def _update_search_nav():
    """Update the search result position label and prev/next state."""
    idxs = _state["filtered_idxs"]
    if idxs is None:
        search_nav.layout.display = "none"
        return
    search_nav.layout.display = "flex"
    current = idx_slider.value
    # Find position of current row in search results
    try:
        pos = idxs.index(current)
        search_pos_label.value = f"&ensp;match <b>{pos + 1}</b> / {len(idxs)}"
    except ValueError:
        search_pos_label.value = f"&ensp;{len(idxs)} matches (current row not in results)"
    search_prev_btn.disabled = (idxs is None or len(idxs) == 0)
    search_next_btn.disabled = (idxs is None or len(idxs) == 0)


def _load_stage(key: tuple[str, str]):
    """Load parquets for the selected stage and update controls."""
    _state["df_a"] = pd.read_parquet(stages_a[key])
    _state["df_b"] = pd.read_parquet(stages_b[key])
    _state["key"] = key
    _state["cfg"] = BENCH_CONFIG[key]
    _state["filtered_idxs"] = None

    max_idx = min(len(_state["df_a"]), len(_state["df_b"])) - 1
    idx_slider.max = max_idx
    idx_input.max = max_idx
    idx_slider.value = 0

    # Update searchable fields dropdown
    search_field.options = _searchable_cols(_state["df_a"], _state["cfg"])
    search_input.value = ""
    search_nav.layout.display = "none"

    # QA summary
    info_output.clear_output(wait=True)
    with info_output:
        da, db = _state["df_a"], _state["df_b"]
        cfg = _state["cfg"]
        print(f"Loaded: {key[0]} / {key[1]}")
        print(f"Rows  A: {len(da):,}   B: {len(db):,}")
        if len(da) != len(db):
            print(f"  WARNING: row count mismatch")
        id_cols = [c for c in cfg["id_cols"] if c in da.columns and c in db.columns]
        if id_cols:
            n = min(len(da), len(db))
            aligned = (da[id_cols].iloc[:n].reset_index(drop=True)
                       .equals(db[id_cols].iloc[:n].reset_index(drop=True)))
            print(f"ID alignment ({id_cols}): {'OK' if aligned else 'MISALIGNED'}")


def _render_current():
    main_output.clear_output(wait=True)
    with main_output:
        render_comparison(
            _state["df_a"], _state["df_b"], _state["cfg"],
            _state["key"], idx_slider.value,
        )
    _update_search_nav()
    save_status.value = ""


# ── Event handlers ────────────────────────────────────────────────────

def _on_stage_change(change):
    _load_stage(change["new"])
    _render_current()

def _on_idx_change(change):
    _render_current()

def _on_search(btn):
    query = search_input.value.strip()
    if not query:
        return
    info_output.clear_output(wait=True)
    with info_output:
        print(f"Searching '{search_field.value}' for: {query} ...")
    hits = search_rows(_state["df_a"], _state["df_b"], _state["cfg"],
                       search_field.value, query)
    _state["filtered_idxs"] = hits
    info_output.clear_output(wait=True)
    with info_output:
        print(f"Search: {len(hits)} matches for '{query}' in '{search_field.value}'")
    if hits:
        idx_slider.value = hits[0]
    _update_search_nav()

def _on_clear(btn):
    _state["filtered_idxs"] = None
    search_input.value = ""
    search_nav.layout.display = "none"
    info_output.clear_output(wait=True)
    with info_output:
        key = _state["key"]
        print(f"Loaded: {key[0]} / {key[1]}  —  filter cleared")
        print(f"Rows  A: {len(_state['df_a']):,}   B: {len(_state['df_b']):,}")

def _on_search_prev(btn):
    idxs = _state["filtered_idxs"]
    if not idxs:
        return
    current = idx_slider.value
    # Find the largest idx in hits that is < current
    prev_hits = [i for i in idxs if i < current]
    idx_slider.value = prev_hits[-1] if prev_hits else idxs[-1]  # wrap around

def _on_search_next(btn):
    idxs = _state["filtered_idxs"]
    if not idxs:
        return
    current = idx_slider.value
    next_hits = [i for i in idxs if i > current]
    idx_slider.value = next_hits[0] if next_hits else idxs[0]  # wrap around

def _on_save(btn):
    record = build_record(
        _state["df_a"], _state["df_b"], _state["cfg"],
        _state["key"], idx_slider.value,
    )
    benchmark, stage = _state["key"]
    out_dir = Path("saved_comparisons")
    out_dir.mkdir(exist_ok=True)
    fname = f"{benchmark}_{stage}_row{idx_slider.value}.json"
    out_path = out_dir / fname
    with open(out_path, "w") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)
    save_status.value = f'<span style="color:green;">Saved: {out_path}</span>'

# Wire up
stage_dropdown.observe(_on_stage_change, names="value")
idx_slider.observe(_on_idx_change, names="value")
search_btn.on_click(_on_search)
clear_btn.on_click(_on_clear)
search_prev_btn.on_click(_on_search_prev)
search_next_btn.on_click(_on_search_next)
save_btn.on_click(_on_save)

# Also trigger search on Enter key in the search input
search_input.on_submit(lambda w: _on_search(None))

# ── Layout ────────────────────────────────────────────────────────────
display(widgets.VBox([
    stage_dropdown,
    widgets.HBox([search_field, search_input, search_btn, clear_btn]),
    search_nav,
    widgets.HBox([idx_slider, idx_input]),
    widgets.HBox([save_btn, save_status]),
    info_output,
    main_output,
]))

# Initial load
_load_stage(available[0])
_render_current()

/tmp/ipykernel_475748/2883001723.py:200: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  search_input.on_submit(lambda w: _on_search(None))


## Usage Notes

1. **Set run paths** in the Configuration cell (`RUN_A`, `RUN_B`). These should be `eval_all` run roots at the timestamp level, e.g.:
   ```
   /share/pierson/matt/UAIR/multirun/2026-03-28_eval_all/00-21-18
   ```
   The notebook auto-resolves the `/0/` multirun subdirectory.

2. **Stage dropdown** lists all benchmark/stage combinations present in both runs. Switching stages reloads the data.

3. **Search** — select a field (prompt, generated_text, vignette, ground_truth, case_id, etc.) and enter a regex pattern. Matches are case-insensitive and search across both A and B completions. Use **Prev/Next** to navigate through matches, or press Enter to search.

4. **Save to JSON** — exports the current row's full comparison (prompt, both completions, ground truth, predictions, usage stats) to `saved_comparisons/{benchmark}_{stage}_row{N}.json` relative to the notebook directory.

5. **Inference vs parse outputs**: Both `llm_inference_*` (raw completions) and `parse_responses_*` (with parsed predictions) are available. The parse stages include the `prediction` column alongside the raw `generated_text`.

6. The notebook assumes rows are in the **same order** across both runs (same dataset, same split). If IDs are misaligned, a warning is printed.